In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
import time
import json
import os
from datetime import datetime
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore")
from aeon.classification.convolution_based import (
    RocketClassifier,
    HydraClassifier,
)
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.feature_based import Catch22Classifier

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [ ]:
#!pip install aeon==0.11.0

In [ ]:
def get_models():
    """Initialize all models from both libraries"""
    # Aeon Direct Classifiers
    aeon_classifiers = {
        "Rocket": RocketClassifier(random_state=42),
        "MiniRocket": RocketClassifier(rocket_transform="minirocket", random_state=42),
        "QUANT": QUANTClassifier(random_state=42),
        "Hydra": HydraClassifier(random_state=42),
        "Catch22": Catch22Classifier(random_state=42),
    }

    return aeon_classifiers


def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset
    Returns X_train, y_train, X_test, y_test
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    return X_train, y_train, X_test, y_test


def get_model_size(model):
    """
    Estimate the size of the model in bytes
    """
    import sys
    import pickle

    return sys.getsizeof(pickle.dumps(model))


def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a model, return detailed metrics including timing
    """
    # Measure fit time
    start_fit_time = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start_fit_time

    # Measure prediction time for training data
    start_pred_train_time = time.time()
    y_pred_train = model.predict(X_train)
    pred_train_time = time.time() - start_pred_train_time

    # Measure prediction time for test data
    start_pred_test_time = time.time()
    y_pred_test = model.predict(X_test)
    pred_test_time = time.time() - start_pred_test_time

    # Calculate performance metrics
    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)

    # Calculate samples per second
    train_samples_per_second = X_train.shape[0] / pred_train_time
    test_samples_per_second = X_test.shape[0] / pred_test_time

    # Get classification report
    class_report = classification_report(y_test, y_pred_test)

    # Store all metrics in a dictionary
    metrics = {
        "model_name": model_name,
        "train_accuracy": float(train_accuracy),
        "test_accuracy": float(test_accuracy),
        "fit_time": float(fit_time),
        "pred_train_time": float(pred_train_time),
        "pred_test_time": float(pred_test_time),
        "train_samples_per_second": float(train_samples_per_second),
        "test_samples_per_second": float(test_samples_per_second),
        "classification_report": class_report,
        "model_size_bytes": get_model_size(model),
    }

    return metrics


def run_experiments(dataset_paths):
    """Run experiments on all datasets with all models from both libraries"""
    models = get_models()
    all_results = defaultdict(list)

    for dataset_path in dataset_paths:
        dataset_name = dataset_path.split("/")[-1].split(".")[0]
        # Load and prepare data
        print(f"\nProcessing dataset: {dataset_name}")
        X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)

        # Group models by library for organized processing
        library_models = {name: model for name, model in models.items()}

        for model_name, model in library_models.items():
            print(f"\nTraining {model_name}...")
            try:
                result = evaluate_model(
                    model,
                    X_train,
                    X_test,
                    y_train,
                    y_test,
                    model_name,
                )
                all_results[dataset_name].append(result)

                print(f"Model: {model_name}")
                print(f"Training Accuracy: {result['train_accuracy']:.4f}")
                print(f"Test Accuracy: {result['test_accuracy']:.4f}")
                print(f"Fit Time: {result['fit_time']:.4f} seconds")
                print("-" * 50)
            except Exception as e:
                print(f"Error with {model_name}: {str(e)}")
                continue

    return all_results


def save_results(results, output_dir="comparison_results"):
    """Save results with separate sections for each library"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = os.path.join(output_dir, f"comparison_results_{timestamp}")

    # Save JSON
    json_filename = f"{base_filename}.json"
    with open(json_filename, "w") as f:
        json_results = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "results": results,
        }
        json.dump(json_results, f, indent=4)

    # Save TXT with organized sections
    txt_filename = f"{base_filename}.txt"
    with open(txt_filename, "w") as f:
        f.write("Time Series Classification Comparison Results\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 100 + "\n\n")

        for dataset_name, dataset_results in results.items():
            f.write(f"\nDataset: {dataset_name}\n")
            f.write("=" * 50 + "\n")

            # Separate results by library
            aeon_results = [
                r for r in dataset_results if r["model_name"].startswith("Aeon")
            ]

            # Write Aeon results
            f.write("\nAeon Results:\n")
            f.write("-" * 30 + "\n")
            for result in aeon_results:
                write_model_results(f, result)

    return json_filename, txt_filename


def write_model_results(f, result):
    """Helper function to write model results"""
    f.write(f"\nModel: {result['model_name']}\n")
    f.write("=" * (len(result["model_name"]) + 7) + "\n")
    f.write(f"Training Accuracy: {result['train_accuracy']:.4f}\n")
    f.write(f"Test Accuracy: {result['test_accuracy']:.4f}\n")
    f.write(f"Fit Time: {result['fit_time']:.4f} seconds\n")
    f.write(f"Training Prediction Time: {result['pred_train_time']:.4f} seconds\n")
    f.write(f"Test Prediction Time: {result['pred_test_time']:.4f} seconds\n")
    f.write(f"Training Samples/Second: {result['train_samples_per_second']:.2f}\n")
    f.write(f"Test Samples/Second: {result['test_samples_per_second']:.2f}\n")
    f.write("\nClassification Report:\n")
    f.write(f"Model Size: {result['model_size_bytes'] / 1024:.2f} KB\n")
    f.write(result["classification_report"])
    f.write("\n" + "-" * 50 + "\n")

CPU times: user 10 μs, sys: 1e+03 ns, total: 11 μs
Wall time: 13.1 μs


In [4]:
# Example dataset paths - replace with your actual paths
dataset_paths = [
    "../data/raw/CounterMovementJump.npy",
    "../data/raw/MP8.npy",
    "../data/raw/MP50.npy",
    "../data/raw/synth_2lines.npy",
]

# Run experiments
results = run_experiments(dataset_paths)

# Save results
json_file, txt_file = save_results(results)

print("\nExperiments completed successfully!")
print(f"Results saved to:\n{json_file}\n{txt_file}")


Processing dataset: CounterMovementJump
X_train shape: (419, 3, 384)
X_test shape: (179, 3, 384)
y_train shape: (419,)
y_test shape: (179,)

Training Rocket...
Model: Rocket
Training Accuracy: 1.0000
Test Accuracy: 0.9497
Fit Time: 17.6663 seconds
--------------------------------------------------

Training MiniRocket...
Model: MiniRocket
Training Accuracy: 1.0000
Test Accuracy: 0.9441
Fit Time: 2.2576 seconds
--------------------------------------------------

Training QUANT...
Model: QUANT
Training Accuracy: 1.0000
Test Accuracy: 0.9330
Fit Time: 2.6544 seconds
--------------------------------------------------

Training Hydra...
Model: Hydra
Training Accuracy: 1.0000
Test Accuracy: 0.9441
Fit Time: 6.0045 seconds
--------------------------------------------------

Training Catch22...
Model: Catch22
Training Accuracy: 1.0000
Test Accuracy: 0.9218
Fit Time: 19.7846 seconds
--------------------------------------------------

Processing dataset: MP8
X_train shape: (1426, 8, 161)
X_test